In [1]:
!pip uninstall -y transformers
!pip install transformers==4.44.2 datasets==2.19.0

Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)


In [2]:
import os
import math
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    RobertaConfig, RobertaForMaskedLM, RobertaTokenizerFast,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback, pipeline
)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
BASE_DIR = "/content/drive/MyDrive/RoBerta_Ancient_Rus_V1"
TOKENIZER_DIR = f"{BASE_DIR}/ancient_rus_tokenizer_BPE"
DATA_FILE = f"{BASE_DIR}/ancient_rus_ready_for_bert.txt"
MODEL_DIR = f"{BASE_DIR}/mini_robert_ancient_rus"

In [7]:
tokenizer = RobertaTokenizerFast.from_pretrained(
        TOKENIZER_DIR,
        max_len=512,
        clean_up_tokenization_spaces=True
    )

In [8]:
special_tokens_dict = {
        'additional_special_tokens': [
            "[CTX_CHURCH]", "[CTX_DAILY]", "[CTX_LEGAL]",
            "[CTX_LIT]", "[CTX_EPIC]", "[CTX_SCIENCE]"
        ]
    }

In [9]:
tokenizer.add_special_tokens(special_tokens_dict)

6

In [11]:
dataset = load_dataset("text", data_files={"train": DATA_FILE})
split_dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

Generating train split: 0 examples [00:00, ? examples/s]

In [12]:
def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=False)

In [13]:
tokenized_datasets = split_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/465075 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (619 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/24478 [00:00<?, ? examples/s]

In [14]:
def group_texts(examples):
      block_size = 256
      concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
      total_length = len(concatenated_examples[list(examples.keys())[0]])
      total_length = (total_length // block_size) * block_size
      return {
          k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
          for k, t in concatenated_examples.items()
      }

In [15]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

Map:   0%|          | 0/465075 [00:00<?, ? examples/s]

Map:   0%|          | 0/24478 [00:00<?, ? examples/s]

In [16]:
config = RobertaConfig(
    vocab_size=len(tokenizer),
    hidden_size=512,
    num_hidden_layers=6,
    num_attention_heads=8,
    intermediate_size=2048,
    max_position_embeddings=514, # У RoBERTa +2 к размеру блока
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    type_vocab_size=1
)

In [17]:
model = RobertaForMaskedLM(config)
model.resize_token_embeddings(len(tokenizer))
print(f"🧠 Параметры Mini-RoBERTa: {model.num_parameters():,}")

🧠 Параметры Mini-RoBERTa: 27,137,688


In [18]:
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return torch.topk(logits, k=5, dim=-1).indices

In [19]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    mask = labels != -100
    labels = labels[mask]
    preds = preds[mask]

    top1_acc = np.mean(preds[:, 0] == labels)
    top3_acc = np.mean(np.any(preds[:, :3] == labels[:, None], axis=1))
    top5_acc = np.mean(np.any(preds[:, :5] == labels[:, None], axis=1))

    return {
        "top1_accuracy": top1_acc,
        "top3_accuracy": top3_acc,
        "top5_accuracy": top5_acc,
    }

In [20]:
class SmartPrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step % 400 == 0:
            print(f"\n🔮 --- ПРОВЕРКА НА ШАГЕ {state.global_step} ---")
            samples = [
                "[CTX_CHURCH] во имѧ ѿц҃а и <mask> и ст҃го дх҃а",
                "[CTX_DAILY] поклоно ѿ онѳима ко <mask>",
                "[CTX_LEGAL] а посулов бояром не <mask>",
                "[CTX_LIT] не лѣпо ли ны бяшетъ братие начяти старыми <mask> трудную повѣсть",
                "[CTX_EPIC] гой еси ты добрый <mask>",
                "[CTX_SCIENCE] а ѿ тоя болезни дай ему пити <mask>"
            ]
            device = kwargs['model'].device
            kwargs['model'].eval()
            with torch.no_grad():
                for text in samples:
                    inputs = tokenizer(text, return_tensors="pt").to(device)
                    mask_idx = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
                    if len(mask_idx) > 0:
                        outputs = kwargs['model'](**inputs)
                        top_3 = torch.topk(outputs.logits[0, mask_idx, :], 3, dim=1).indices[0].tolist()
                        decoded = [tokenizer.decode([t]).replace("Ġ", "").strip() for t in top_3]
                        cat = text.split(']')[0] + ']'
                        clean_text = text.replace(cat, '').strip()
                        print(f"📝 {cat:<13} | {clean_text}  ->  {decoded}")
            kwargs['model'].train()
            print("----------------------------------------------\n")

In [22]:
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    overwrite_output_dir=True,
    num_train_epochs=15,
    per_device_train_batch_size=64,
    gradient_accumulation_steps=2,
    evaluation_strategy="steps",
    eval_steps=400,
    save_steps=400,
    save_total_limit=2,
    logging_steps=100,
    prediction_loss_only=False, # Обязательно False для метрик!
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=1000,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2, # В Colab 2 воркера работают отлично
    report_to="none",
    load_best_model_at_end=True
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
    tokenizer=tokenizer,
    callbacks=[SmartPrinterCallback()],
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

In [24]:
trainer.train()

Step,Training Loss,Validation Loss,Top1 Accuracy,Top3 Accuracy,Top5 Accuracy
400,7.281700,7.193415,0.114592,0.172220,0.201809
800,7.011300,6.946617,0.136148,0.195356,0.225143
1200,6.286500,6.118277,0.200518,0.260919,0.288278
1600,5.262200,5.026170,0.255451,0.328583,0.365264
2000,4.520000,4.355517,0.302607,0.398549,0.443367
2400,4.022700,3.857926,0.356137,0.466435,0.514754



🔮 --- ПРОВЕРКА НА ШАГЕ 400 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и <mask> и ст҃го дх҃а  ->  ['҃', 'и', 'не']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко <mask>  ->  ['.', 'и', ',']
📝 [CTX_LEGAL]   | а посулов бояром не <mask>  ->  ['а', 'и', '.']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми <mask> трудную повѣсть  ->  ['и', '.', ',']
📝 [CTX_EPIC]    | гой еси ты добрый <mask>  ->  ['.', 'и', ',']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити <mask>  ->  ['а', '.', ',']
----------------------------------------------


🔮 --- ПРОВЕРКА НА ШАГЕ 400 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и <mask> и ст҃го дх҃а  ->  ['҃', 'и', 'не']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко <mask>  ->  ['.', 'и', ',']
📝 [CTX_LEGAL]   | а посулов бояром не <mask>  ->  ['а', 'и', '.']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми <mask> трудную повѣсть  ->  ['и', '.', ',']
📝 [CTX_EPIC]    | гой еси ты добрый <mask>  ->  ['.', 'и', ',']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити <mask>  ->  ['а'

Step,Training Loss,Validation Loss,Top1 Accuracy,Top3 Accuracy,Top5 Accuracy
400,7.281700,7.193415,0.114592,0.172220,0.201809
800,7.011300,6.946617,0.136148,0.195356,0.225143
1200,6.286500,6.118277,0.200518,0.260919,0.288278
1600,5.262200,5.026170,0.255451,0.328583,0.365264
2000,4.520000,4.355517,0.302607,0.398549,0.443367
2400,4.022700,3.857926,0.356137,0.466435,0.514754
2800,3.635200,3.471591,0.401865,0.521465,0.572284
3200,3.347600,3.190257,0.440514,0.561818,0.612372
3600,3.140400,2.968699,0.474387,0.594592,0.642471
4000,2.955700,2.796639,0.499607,0.617820,0.664896



🔮 --- ПРОВЕРКА НА ШАГЕ 2800 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и <mask> и ст҃го дх҃а  ->  ['села', 'положиша', 'крѣпости']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко <mask>  ->  ['мне', 'попу', 'матери']
📝 [CTX_LEGAL]   | а посулов бояром не <mask>  ->  ['имати', 'быти', '.']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми <mask> трудную повѣсть  ->  ['и', 'на', 'своею']
📝 [CTX_EPIC]    | гой еси ты добрый <mask>  ->  ['молодец', 'конь', '!']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити <mask>  ->  ['.', ':', '?']
----------------------------------------------


🔮 --- ПРОВЕРКА НА ШАГЕ 2800 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и <mask> и ст҃го дх҃а  ->  ['села', 'положиша', 'крѣпости']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко <mask>  ->  ['мне', 'попу', 'матери']
📝 [CTX_LEGAL]   | а посулов бояром не <mask>  ->  ['имати', 'быти', '.']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми <mask> трудную повѣсть  ->  ['и', 'на', 'своею']
📝 [CTX_EPIC]    | гой еси ты добрый <mas

There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias'].


TrainOutput(global_step=6150, training_loss=4.184296800566883, metrics={'train_runtime': 4230.0621, 'train_samples_per_second': 186.295, 'train_steps_per_second': 1.454, 'total_flos': 2.3204925494329344e+16, 'train_loss': 4.184296800566883, 'epoch': 14.981729598051157})

In [25]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

('/content/drive/MyDrive/RoBerta_Ancient_Rus_V1/mini_robert_ancient_rus/tokenizer_config.json',
 '/content/drive/MyDrive/RoBerta_Ancient_Rus_V1/mini_robert_ancient_rus/special_tokens_map.json',
 '/content/drive/MyDrive/RoBerta_Ancient_Rus_V1/mini_robert_ancient_rus/vocab.json',
 '/content/drive/MyDrive/RoBerta_Ancient_Rus_V1/mini_robert_ancient_rus/merges.txt',
 '/content/drive/MyDrive/RoBerta_Ancient_Rus_V1/mini_robert_ancient_rus/added_tokens.json',
 '/content/drive/MyDrive/RoBerta_Ancient_Rus_V1/mini_robert_ancient_rus/tokenizer.json')

In [26]:
eval_results = trainer.evaluate()
print(f"\n📊 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
print(f"Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")
print(f"Top-1 Точность: {eval_results.get('eval_top1_accuracy', 0):.2%}")
print(f"Top-3 Точность: {eval_results.get('eval_top3_accuracy', 0):.2%}")


📊 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:
Loss: 2.5294
Perplexity: 12.55
Top-1 Точность: 54.44%
Top-3 Точность: 65.77%


In [27]:
# Загружаем через пайплайн
fill_mask = pipeline(
    "fill-mask",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    device=0, # Использовать GPU
)

# Усложненные тесты для проверки генерализации
hard_tests = [
    {
        "category": "⛪️ [CTX_CHURCH] (Тест на учеников/апостолов в дательном падеже)",
        "text": "[CTX_CHURCH] И рече господь къ <mask> своимъ, глаголя..."
    },
    {
        "category": "🏡 [CTX_DAILY] (Тест на родственные связи и долги)",
        "text": "[CTX_DAILY] Поклонъ ѿ петра ко <mask> . а серебро ми отдай."
    },
    {
        "category": "⚖️ [CTX_LEGAL] (Тест на Русскую Правду: кого убили?)",
        "text": "[CTX_LEGAL] Аже кто оубиеть <mask> , то платити виру 40 гривенъ."
    },
    {
        "category": "📚 [CTX_LIT] (Тест на летописи: на какую землю пошел князь?)",
        "text": "[CTX_LIT] И пошелъ князь игорь на <mask> землю со своею дружиною."
    },
    {
        "category": "⚔️ [CTX_EPIC] (Тест на предложный падеж: на чем выезжал?)",
        "text": "[CTX_EPIC] Выезжал добрый молодец из города на добромъ <mask> ."
    },
    {
        "category": "🌿 [CTX_SCIENCE] (Тест на лечебники: что пить?)",
        "text": "[CTX_SCIENCE] Аще кто боленъ главою, дай пити ему <mask> от травы."
    }
]

print("\n" + "=" * 70)
print("🎓 ФИНАЛЬНЫЙ ЭКЗАМЕН MINI-ROBERTA (УСЛОЖНЕННЫЕ КОНТЕКСТЫ)")
print("=" * 70)

for test in hard_tests:
    print(f"\n🔹 {test['category']}")
    print(f"Текст: {test['text']}")

    # RoBERTa иногда возвращает пробелы перед знаком препинания, pipeline это обработает
    results = fill_mask(test["text"])

    for i, res in enumerate(results[:5]): # Выводим Топ-5, раз уж она так в нем хороша!
        # Очищаем слово от лишних пробелов для красоты
        clean_word = res['token_str'].strip()
        print(f"  {i+1}. {clean_word:<15} (Уверенность: {res['score']*100:.1f}%)")


🎓 ФИНАЛЬНЫЙ ЭКЗАМЕН MINI-ROBERTA (УСЛОЖНЕННЫЕ КОНТЕКСТЫ)

🔹 ⛪️ [CTX_CHURCH] (Тест на учеников/апостолов в дательном падеже)
Текст: [CTX_CHURCH] И рече господь къ <mask> своимъ, глаголя...
  1. людемъ          (Уверенность: 42.6%)
  2. богоу           (Уверенность: 10.2%)
  3. сыномъ          (Уверенность: 6.1%)
  4. лицемъ          (Уверенность: 4.0%)
  5. женѣ            (Уверенность: 2.8%)

🔹 🏡 [CTX_DAILY] (Тест на родственные связи и долги)
Текст: [CTX_DAILY] Поклонъ ѿ петра ко <mask> . а серебро ми отдай.
  1. василью         (Уверенность: 17.7%)
  2. климѧ           (Уверенность: 17.6%)
  3. матьри          (Уверенность: 14.1%)
  4. матери          (Уверенность: 6.2%)
  5. попу            (Уверенность: 5.1%)

🔹 ⚖️ [CTX_LEGAL] (Тест на Русскую Правду: кого убили?)
Текст: [CTX_LEGAL] Аже кто оубиеть <mask> , то платити виру 40 гривенъ.
  1. мужь            (Уверенность: 23.1%)
  2. господинъ       (Уверенность: 3.8%)
  3. то              (Уверенность: 3.4%)
  4. куны            (Ув

In [28]:
HF_USERNAME = "AlexSychovUN"

In [33]:
ROBERTA_DIR = "/content/drive/MyDrive/RoBerta_Ancient_Rus_V1/mini_robert_ancient_rus"

In [35]:
from huggingface_hub import notebook_login

notebook_login()

In [36]:
print("\n🚀 ПУШИМ MINI-ROBERTA (BPE)...")
roberta_model = RobertaForMaskedLM.from_pretrained(ROBERTA_DIR)
roberta_tokenizer = RobertaTokenizerFast.from_pretrained(ROBERTA_DIR)

roberta_repo = f"{HF_USERNAME}/mini-roberta-ancient-rus"
roberta_model.push_to_hub(roberta_repo)
roberta_tokenizer.push_to_hub(roberta_repo)
print(f"✅ RoBERTa успешно загружена: https://huggingface.co/{roberta_repo}")


🚀 ПУШИМ MINI-ROBERTA (BPE)...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...wt1wxbm/model.safetensors:   1%|          |  553kB /  109MB            

README.md: 0.00B [00:00, ?B/s]

✅ RoBERTa успешно загружена: https://huggingface.co/AlexSychovUN/mini-roberta-ancient-rus
